<table>
  <tr>
    <td><div align="left"><font size="30">Robotics Toolbox for Python</font></div></td>
    <td><img src="figs/RobToolBox_RoundLogoB.png" width="300"></td>
  </tr>
</table>

<p></p>
<div align="center" style="font-size: 1.5em;">🤖🚀 Robotics without the cruft</div>
<p></p>


(c) Peter Corke 2026

In [ ]:
try:
    import piplite
    await piplite.install(["roboticstoolbox-python"])
except ImportError:
    pass  # not running in Pyodide (e.g. local Jupyter) -- assume it's already pip-installed

import roboticstoolbox
print("roboticstoolbox version:", roboticstoolbox.__version__)

# import matplotlib.pyplot as plt
import numpy as np
np.set_printoptions(linewidth=100, formatter={'float': lambda x: f"{x:8.3g}" if abs(x) > 1e-10 else f"{0:8.3g}"})

import math

# Mobile robotics

## Mobile robot kinematics

We can create a vehicle with bicycle kinematics, with a wheelbase of 2m.

In [ ]:
from roboticstoolbox import Bicycle, VehicleIcon

veh = Bicycle(L=2, animation=VehicleIcon("redcar", scale=2), workspace=10)
print(veh)

We can find the maximum path curvature it can achieve, which is the reciprocal of the minimum turning radius.

In [ ]:
print(veh.curvature_max)

The initial state of the vehicle $(x,y,\theta)$, where $\theta$ is the heading angle, can be set at contruction time but defaults to zero

In [ ]:
print(veh.x)

The state derivative $(\dot{x}, \dot{y}, \dot{\theta})$ is a function of current state and the inputs $(v, \gamma)$ where $\gamma$ is the angle of the steered wheel.

In [ ]:
xd = veh.deriv(u=(1, 0.2), x= [0,0,0])
print(xd)

which indicates motion in the x-direction and a change of heading angle.

The model supports simple animation of motion directed by a `control` function.  In this case the control sets a constant velocity and a steered wheel angle of 0.5 rad for $1<t<2$

In [ ]:
veh.run(10, control=lambda v, t, x: (1, 0.5 if 1<t<2 else 0), animate=True)

## Graph-based planning

In [ ]:
from pgraph import UGraph # from PGraph package
from roboticstoolbox import rtb_path_to_datafile # from Robotics Toolbox package
import json

# load an example route map from the rtb-data package
with open(rtb_path_to_datafile('data/queensland.json'), 'r') as f:
    data = json.loads(f.read())

g = UGraph() # create an undirected graph
for name, info in data['places'].items():
    g.add_vertex(name=name, coord=info["utm"]) # add places as vertices
for route in data['routes']:
    g.add_edge(route['start'], route['end'], cost=route['distance']) # add routes as edges

g.plot()

In [ ]:
path, length, parents = g.path_Astar('Hughenden', 'Brisbane')

g.plot(block=None)
g.highlight_path(path)

## Occupancy grid path planning

In [ ]:
from roboticstoolbox import rtb_load_matfile, DistanceTransformPlanner

house = rtb_load_matfile('data/house.mat')
floorplan = house['floorplan']
places = house['places']

pmarker = dict(markersize=6, color='y')
dx = DistanceTransformPlanner(floorplan, inflate=5)
dx.plan(places.kitchen)
dx.plot();

In [ ]:
p = dx.query(places.br3)

dx.plot(p, inflated=True, path_marker=pmarker);

## Sample-based planning

In [ ]:
from roboticstoolbox import PRMPlanner


house = rtb_load_matfile('data/house.mat')
floorplan = house['floorplan']
places = house['places']

prm = PRMPlanner(occgrid=floorplan, seed=0)
prm.plan(npoints=50)

prm.plot(edge=dict(alpha=0.3), vertex=dict(alpha=0.3))
print(prm)

In [ ]:
# reseed the PRN to get a workable solution
# prm = PRMPlanner(occgrid=floorplan, seed=2)
prm.plan(npoints=200)
prm.plot(edge=dict(alpha=0.1), vertex=dict(alpha=0.1))
print(prm)

## Planning with motion constraints

### Dubbins paths

In [ ]:
from roboticstoolbox import DubinsPlanner

qs = (0, 0, math.pi/2)
qg = (1, 0, math.pi/2)

dubins = DubinsPlanner(curvature=1)
path, status = dubins.query(qs, qg)

dubins.plot(path);

### Reeds-Shepp path

In [ ]:
from roboticstoolbox import ReedsSheppPlanner

rs = ReedsSheppPlanner(curvature=1)
path, status = rs.query(qs, qg)

rs.plot(path);

### Configuration-space planning

Let's look at the classic piano mover's problem

In [ ]:
from spatialmath import Polygon2
from roboticstoolbox import PolygonMap, RRTPlanner, Bicycle

# start and goal configuration
qs = (2, 8, -math.pi/2)
qg = (8, 2, -math.pi/2)

# obstacle map
map = PolygonMap(workspace=[0, 10])
map.add([(5, 50), (5, 6), (6, 6), (6, 50)])
map.add([(5, 4), (5, -50), (6, -50), (6, 4)])

# create a polygon to represent the piano
length, width  =3, 1.5
piano = Polygon2(
    np.array([(-length / 2, width / 2), (-length / 2, -width / 2), (length / 2, -width / 2), (length / 2, width / 2)]).T
)

# the piano has bicycle kinematics with a maximum steering angle of 1 radian and a wheelbase of 2m (ok, it's an odd piano)
vehicle = Bicycle(steer_max=1, L=2, polygon=piano)

rrt = RRTPlanner(map=map, vehicle=vehicle, verbose=False, npoints=50, showsamples=True, seed=0)

We'll build an RRT with its goal `qg` in the lower-right of the figure

In [ ]:
map.plot()
rrt.plan(goal=qg)

We have found a bunch of piano poses that don't intersect with the red obstacle, a dot represents the position $(x,y)$ and the translucent rectangle indicates its orientation.  Then we joined them up using an RRT.

Now we can query for a path to the goal, given a start pose `qs` which is in the upper left of the figure.

In [ ]:
path, status = rrt.query(start=qs)
print(status)

A path has been found and is described by a set of RRT vertices with pose $(x,y,\theta)$.

In [ ]:
map.plot()
rrt.g.plot(colorcomponents=False, text=False, force2d=True,
    vopt=dict(color='darkblue', marker='o', markersize=10), 
    eopt=dict(color='darkblue', linewidth=3), block=None)
rrt.g.highlight_path(status.vertices, color='r')


The planner returns a smoothed path by fitting Dubins curves between the RRT vertices, resulting in a path that is driveable by the piano

In [ ]:
with np.printoptions(threshold=20):
    print(path)

We can overlay that on the RRT

In [ ]:
map.plot()
rrt.g.plot(colorcomponents=False, text=False, force2d=True,
    vopt=dict(color='darkblue', marker='o', markersize=10), 
    eopt=dict(color='darkblue', linewidth=3), block=None)
rrt.plot(path);

and animate it as a series of snapshots of the piano on its journey

In [ ]:
from roboticstoolbox import VehiclePolygon

va = VehiclePolygon(piano)
map.plot(block=None)
for i in np.unique(np.rint(np.linspace(0, len(path) - 1, 20)).astype(int)):
    va.plot(path[i, :], alpha=0.2)


## Pose-graph optimization

In this example we load the classic "Killian Court" SLAM dataset from a TORO file. The vertices of the graph represent robot pose estimated from wheel odometry.  The edges represent constraints from lidar scan matching of unique landmarks in the scene.

We can load the data and display the vertices (blue circles) and the edges (colored line).

In [ ]:
from roboticstoolbox import PoseGraph

pg = PoseGraph('data/killian-small.toro')

pg.plot(text=False, block=None)


and we can see the effect of odometry pose drift over multiple passes around the corridors.

We can perform pose graph optimization to adjust the vertices so as to bring them into line with the lidar observations.

In [ ]:
pg.optimize()

pg.plot(text=False, block=None)

# Arm robotics

We will load a Denavit-Hartenberg model of the Franka "Panda" robot and display it

In [ ]:
from roboticstoolbox import models

panda = models.DH.Panda()

print(panda)

The model contains a list of links, base and tool transforms, and a set of useful named joint coordinates.

We can display the configuration of the robot at the "ready" coordinate

In [ ]:
q = panda.qr

panda.plot(q)

The pose of the tool tip can be computed, it's an `SE3` object

In [ ]:
panda.fkine(q)

Now let's decide we want the tool tip to be at $(0.5, 0.2, 0.5)$ and pointing straight downwards (Z-axis down). We can easily compute the invere kinematics numerically

In [ ]:
from spatialmath import SE3

sol = panda.ikine_LM(SE3.Trans(0.5, 0.2, 0.5) * SE3.Rx(math.pi) )
print(sol)

and the resulting `sol` object contains the status (yes, we were successful), the joint coordinates, as well as information about the residual and number of iterations and restarts.  We can validate the solution

In [ ]:
q = sol.q
panda.fkine(q)

We can compute a joint-space trajectory between two sets of coordinates

In [ ]:
from roboticstoolbox import jtraj

tg = jtraj(panda.qz, panda.qr, 100)
print(tg)

The `tg` object contains the joint coordinates, joint rates, and joint accelerations as time series.

We can pass a trajectory (100x7 matrix) to plot and the result will be an animation

In [ ]:
panda.plot(tg.q)

and if we pass the trajectory to `fkine` we get an `SE3` object that contains multiple values, which we can slice as discussed earlier today

In [ ]:
T_tg = panda.fkine(tg.q)
print(len(T_tg))
print(T_tg[10])

I've never really liked DH parameters so we can also create a kinematic model using ETS notation

In [ ]:
from roboticstoolbox import models

panda = models.ETS.Panda()
panda

In ETS the model is just one long string of transforms, some constant, some are joint variables

In [ ]:
ets = panda.ets()
print(ets)

which is another list-like sliceable object

In [ ]:
ets[2:4]

I can write an ETS string like that for **any robot**. There are none of the DH rule to get in the way.

Then I can turn that into a robot object

In [ ]:
from roboticstoolbox import Robot

r = Robot(ets)
print(r)

and compute its forward kinematics

In [ ]:
r.fkine(q)

For any robot we can compute Jacobians

In [ ]:
panda.jacob0(panda.qr)

In [ ]:
panda.jacobe(q)

and Hessians

In [ ]:
H = panda.hessian0(q)
H.shape

The robot's manipulability in this configuration is summarized by

In [ ]:
q = panda.qr

panda.manipulability(q)

But we can get a better understanding by displaying the velocity ellipsoid (for translational motion)

In [ ]:
panda.plot(q)
panda.vellipse(q);

We can also import a model from URDF files. Models like `Panda` here are bundled directly with the Toolbox, so they work everywhere -- including in this JupyterLite environment.

Some other URDF models (e.g. `Jaco`, `PR2`, `UR5`, `YuMi`) are instead fetched on first use from the [robot_descriptions](https://github.com/robot-descriptions/robot_descriptions.py) package, which clones a git repository -- this isn't possible in a browser sandbox like JupyterLite, so those specific models will raise an error here (though they work fine in a regular Python environment).

In [ ]:
panda = models.URDF.Panda()
panda

In [ ]:
# panda.plot(q, backend="swift")

To show the dynamics capability we will load one of the oldest models in the toolbox which has quite an accurate dynamic model.

In [ ]:
puma = models.DH.Puma560()
print(puma)

We see that it is tagged with "dynamics" at the top of the tables above.

The dynamic parameters, rigid-body and friction, are

In [ ]:
puma.dynamics()

For a notional pose (not singular) the rigid-body matrix terms can be easily computed

In [ ]:
q = puma.qn

puma.inertia(q)

In [ ]:
puma.gravload(q)

In [ ]:
puma.coriolis(q, [1, 0, 0, 0,0, 0])

Or, in task space they are

In [ ]:
puma.inertia_x(q)

In [ ]:
puma.gravload_x(q)

# What next for Robotics Toolbox?

* play nicely with other packages: Coal, OMPL, ikfast/ssik, Rerun, Pinocchio, CoppeliaSim